In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def plot_cumulative_column(df, column, time_column, time_ranges, labels, colors, title, save_path, xlim=None):
    """
    General function to generate and save a cumulative probability plot (as points)
    for a specified column filtered by time intervals.

    Parameters:
    - df: DataFrame containing the data.
    - column: Name of the column to analyze.
    - time_column: Name of the column indicating time (e.g., 'event start time (ms)').
    - time_ranges: List of (start, end) tuples for filtering.
    - labels: List of labels for each time range.
    - colors: List of colors for each curve.
    - title: Title of the plot.
    - save_path: Where to save the PNG.
    - xlim: Tuple of (min, max) for x-axis limits.
    """
    plt.figure(figsize=(10, 6))

    for (start, end), label, color in zip(time_ranges, labels, colors):
        data = df[(df[time_column] >= start) & (df[time_column] <= end)][column].dropna()
        if len(data) == 0:
            continue
        sorted_data = np.sort(data)
        cum_prob = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
        plt.scatter(sorted_data, cum_prob, label=label, color=color, s=10)

    plt.xlabel(column)
    plt.ylabel("Cumulative Probability")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    if xlim:
        plt.xlim(xlim)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Saved: {save_path}")

def process_directory_for_plots(directory_path, output_dir=None):
    """
    Processes all CSV files in a directory and generates cumulative plots for:
    - 'peak amp (pA)'
    - 'Area (pA*ms)'
    - Converted 'Ints. Freq (Hz)' → interevent interval (ms)

    Each plot includes data from:
    - 0–300,000 ms (black)
    - 330,000–660,000 ms (red)
    - 690,000–990,000 ms (blue)
    """
    if output_dir is None:
        output_dir = directory_path

    os.makedirs(output_dir, exist_ok=True)

    csv_files = glob.glob(os.path.join(directory_path, "*.csv"))
    if not csv_files:
        print("No CSV files found in the directory.")
        return

    time_ranges = [
        (0, 300000),
        (330000, 660000),
        (690000, 990000)
    ]
    labels = [
        "0–300,000 ms",
        "330,000–660,000 ms",
        "690,000–990,000 ms"
    ]
    colors = ["black", "red", "blue"]

    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file)
            base_name = os.path.splitext(os.path.basename(csv_file))[0]

            # Peak Amplitude Plot
            save_path_amp = os.path.join(output_dir, f"{base_name}_peak_amp.png")
            plot_cumulative_column(
                df,
                column="peak amp (pA)",
                time_column="event start time (ms)",
                time_ranges=time_ranges,
                labels=labels,
                colors=colors,
                title=f"Cumulative Probability of Peak Amplitude\n{base_name}",
                save_path=save_path_amp,
                xlim=(-100, 0)
            )

            # Area Plot
            save_path_area = os.path.join(output_dir, f"{base_name}_area.png")
            plot_cumulative_column(
                df,
                column="Area (pA*ms)",
                time_column="event start time (ms)",
                time_ranges=time_ranges,
                labels=labels,
                colors=colors,
                title=f"Cumulative Probability of Event Area\n{base_name}",
                save_path=save_path_area,
                xlim=(-500, 0)
            )

            # Transformed Inst. Freq. (Hz) → Interevent Interval (ms)
            transformed_df = df.copy()
            if "Inst. Freq. (Hz)" in transformed_df.columns:
                transformed_df["interevent interval (ms)"] = 1000 / transformed_df["Inst. Freq. (Hz)"]
                save_path_iei = os.path.join(output_dir, f"{base_name}_interevent_interval.png")
                plot_cumulative_column(
                    transformed_df,
                    column="interevent interval (ms)",
                    time_column="event start time (ms)",
                    time_ranges=time_ranges,
                    labels=labels,
                    colors=colors,
                    title=f"Cumulative Probability of Interevent Interval\n{base_name}",
                    save_path=save_path_iei
                    # Optionally add xlim=(0, some upper limit)
                )

        except Exception as e:
            print(f"Error processing {csv_file}: {e}")

# Example usage:
# process_directory_for_plots("path/to/csv_directory", output_dir="path/to/save_plots")

In [ ]:
# Example usage:
process_directory_for_plots("./data/ManualGranuleAnalysis_10_July_csvs", output_dir="./data/ManualGranuleAnalysis_10_July_csvs")